In [1]:
import findspark
findspark.init()

import os
import json
import pymongo
import pandas as pd

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
# %%
# -----------------------------
# Connection configuration
# -----------------------------
mysql_args = {
    "host_name": "localhost",
    "port": "3306",
    "db_name": "adventureworks",
    "conn_props": {
        "user": "root",
        "password": "#Hi10172004",
        "driver": "com.mysql.cj.jdbc.Driver"
    }
}

mongodb_args = {
    "db_name": "adventureworks",
    "collection": "dim_customers_vw"
}
base_dir = os.path.join(os.getcwd(), "proj_data")
stream_dir = os.path.join(base_dir, "streaming", "sales_orders")
batch_dir = os.path.join(base_dir, "batch")

employee_csv = os.path.join(batch_dir, "dim_employee.csv")

bronze_dir = os.path.join(base_dir, "bronze")
silver_dir = os.path.join(base_dir, "silver")

In [3]:
# %%
def get_mysql_dataframe(spark_session, sql_query: str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"

    return (
        spark_session.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", args['conn_props']['driver'])
        .option("user", args['conn_props']['user'])
        .option("password", args['conn_props']['password'])
        .option("query", sql_query)
        .load()
    )

In [4]:
# %%
# -----------------------------
# Create Spark session
# -----------------------------
mysql_spark_jar = os.path.join(
    os.getcwd(),
    "mysql-connector-j-9.1.0",
    "mysql-connector-j-9.1.0.jar"
)
'''
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("AdventureWorks Final Project")
    .config("spark.jars", mysql_spark_jar)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)'''

spark = (
    SparkSession.builder
    .appName("AdventureWorks Data Lakehouse")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0"
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(mysql_spark_jar)
print(os.path.exists(mysql_spark_jar))


C:\Users\aadit\Downloads\dssystems\DS-2002\FinalProj\mysql-connector-j-9.1.0\mysql-connector-j-9.1.0.jar
True


In [5]:
spark

In [ ]:
# %%
# -----------------------------
# Load dimensions
# -----------------------------

# MySQL dimensions
sql_dim_date = "SELECT * FROM adventureworks.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

sql_dim_products = "SELECT * FROM adventureworks.dim_products_vw"
df_dim_products = get_mysql_dataframe(spark, sql_dim_products, **mysql_args)

In [ ]:
# Mongo dimension
mongo_uri = "mongodb://localhost:27017/"

client = pymongo.MongoClient(mongo_uri)
db = client["adventureworks"]

# Load JSON file into MongoDB collection
customers_json = os.path.join(batch_dir, "dim_customers.json")

with open(customers_json, "r") as f:
    customer_docs = json.load(f)

# Recreate collection for idempotency
db.drop_collection("dim_customers_vw")
db["dim_customers_vw"].insert_many(customer_docs)

client.close()

# Fetch Mongo collection as Spark DataFrame
spark.conf.set("spark.mongodb.input.uri", mongo_uri)

df_dim_customers = (
    spark.read.format("com.mongodb.spark.sql.DefaultSource")
    .option("database", "adventureworks")
    .option("collection", "dim_customers_vw")
    .load()
    .drop("_id")
)



In [7]:
spark.stop()